# L11 · 双均线择时（DualMA）

**预计时长**：75 min | **难度**：⭐⭐⭐ | **前置**：Phase 1 全部

## 本节目标
1. 双均线策略的金融逻辑：金叉 / 死叉 / 长短周期博弈
2. 手写向量化信号生成（无 for 循环）
3. 回测并对比 qtrader 框架版的 `DualMAStrategy`
4. 参数敏感性：short/long 组合对夏普的影响

## 第 1 段：金融概念

### 双均线（Moving Average Crossover）核心逻辑
| 信号 | 条件 | 含义 |
|------|------|------|
| **金叉** | MA_short 从下方上穿 MA_long | 短期上涨动能强于长期 → 买入 |
| **死叉** | MA_short 从上方下穿 MA_long | 短期下跌动能强于长期 → 卖出 |
| 震荡 | MA 交织反复 | 双均线失效区，常见于横盘 |

### 为什么有效（也为什么失效）
- **有效**：趋势跟踪，能在持续上涨/下跌中捕捉大部分中段利润
- **失效**：震荡市频繁假信号，每次假金叉都吃 0.3% 双边成本
- **A 股适配**：T+1 制度下当日买入不能卖，所以信号必须 `shift(1)`——用昨日信号决定今日仓位

### 参数选择经验
| 短/长 | 性格 | 适用场景 |
|-------|------|---------|
| 5/20 | 敏捷 | 短线，交易频繁 |
| 10/30 | 平衡 | 中线主流 |
| 20/60 | 保守 | 大波段，过滤噪音 |
| 5/60 | 跨频 | 短长周期博弈，信号少但强 |

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation（共享 _data/_style）+ project root
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (
    _cwd / 'learning' / 'phase1_foundation'
    if (_cwd / 'learning' / 'phase1_foundation' / '_data.py').exists()
    else _cwd.parent / 'phase1_foundation'
)
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data
from qtrader import (
    BacktestEngine, CostModel,
    DualMAStrategy, TurtleStrategy, GridStrategy,
    print_comparison_table,
)

## 第 2 段：手写双均线信号

**关键技巧**：避免未来函数——信号用「昨日 MA 比较」决定「今日仓位」。

In [ ]:
byd = get_stock_data('002594').set_index('date')

def dual_ma_signal(close: pd.Series, short: int = 5, long: int = 20) -> pd.Series:
    \"\"\"手写双均线信号（向量化，无 for 循环）。

    返回 [0, 1] 目标仓位序列，index 与 close 对齐。
    \"\"\"
    ma_s = close.rolling(short).mean()
    ma_l = close.rolling(long).mean()
    # 金叉后持有，死叉后空仓；shift(1) 避免未来函数
    signal = (ma_s > ma_l).astype(int).shift(1).fillna(0)
    return signal

sig = dual_ma_signal(byd['close'], short=5, long=20)
print(f"持仓天数占比: {sig.mean():.2%}")
print(f"信号切换次数: {(sig.diff().abs() > 1e-6).sum()}")

# 可视化：价格 + 双均线 + 持仓阴影
fig, ax = plt.subplots(figsize=(13, 5))
byd['close'].plot(ax=ax, label='收盘价', alpha=0.7)
byd['close'].rolling(5).mean().plot(ax=ax, label='MA5', linestyle='--')
byd['close'].rolling(20).mean().plot(ax=ax, label='MA20', linestyle='--')
ax.fill_between(sig.index, byd['close'].min(), byd['close'].max(),
                where=(sig == 1), alpha=0.1, color='green', label='持仓区')
ax.set_title('比亚迪 + 双均线 + 持仓区（5/20 参数）')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 第 3 段：向量化回测 + 对比 qtrader 框架版

手写简化版回测 → 对比 `qtrader.DualMAStrategy`，应得到几乎一致的结果。

In [ ]:
# 手写回测
rets = byd['close'].pct_change().fillna(0)
turnover = sig.diff().abs().fillna(0)
cost_rate = 0.00094  # 双边总费率（佣金+印花+过户+滑点）
strat_nav = (1 + sig.shift(1).fillna(0) * rets - turnover * cost_rate).cumprod()
bench_nav = (1 + rets).cumprod()

# qtrader 框架版（同样 5/20 参数 + 默认 CostModel）
engine = BacktestEngine()  # 默认 A 股 2026 真实成本
result = engine.run(byd.reset_index(), DualMAStrategy(short=5, long=20))

fig, ax = plt.subplots(figsize=(13, 5))
bench_nav.plot(ax=ax, label='买入持有', alpha=0.5)
strat_nav.plot(ax=ax, label='手写双均线', linestyle='--')
result.nav.plot(ax=ax, label='qtrader.DualMAStrategy', linewidth=1.5)
ax.set_title('手写 vs 框架版（应几乎重合）')
ax.legend()
plt.tight_layout(); plt.show()

print(f"手写版终值:  {strat_nav.iloc[-1]:.3f}")
print(f"框架版终值:  {result.nav.iloc[-1]:.3f}")
print(f"差异来源: 手写版用简化成本率 0.00094，框架版按买卖方向分别计征")

## 第 4 段：参数敏感性

哪个 short/long 组合最优？这正是「过拟合」最容易出现的场景——
先跑一遍网格，再用 walk-forward 验证（下一段）。

In [ ]:
from qtrader.optimize import grid_search, walk_forward

param_grid = {'short': [3, 5, 10], 'long': [20, 30, 60]}
top = grid_search(engine, byd.reset_index(), DualMAStrategy, param_grid,
                  metric='sharpe', top_n=5)
print("Top 5 参数组合（按夏普降序）：")
print(top[['short', 'long', 'sharpe', 'sortino', 'max_drawdown', 'n_trades']].to_string(index=False))

## 第 5 段：Walk-Forward 验证（防过拟合）

网格搜索选出来的「最优参数」在**未来**还成立吗？切分训练/测试段验证。

In [ ]:
wf = walk_forward(engine, byd.reset_index(), DualMAStrategy,
                  param_grid, train_ratio=0.7, metric='sharpe')
print(f"训练段最优参数: {wf.best_params}")
print(f"训练段 Sharpe: {wf.train_metric:.3f}")
print(f"测试段 Sharpe: {wf.test_metric:.3f}")
print(f"过拟合差距:    {wf.overfit_gap:+.3f}")
if wf.overfit_gap > 0.5:
    print("⚠️ 训练-测试差距 > 0.5，参数大概率过拟合")
elif wf.overfit_gap < 0:
    print("✓ 测试段反而更好，参数稳健")
else:
    print("○ 差距合理，参数有一定泛化能力")

## 第 6 段：本节要点

1. **双均线本质** = 趋势跟踪，震荡市失效
2. **shift(1) 必须有**，否则未来函数 → 夏普虚高 3-10 倍
3. **手写 vs 框架版** 应几乎重合，差异只在成本模型精细度
4. **网格搜索 ≠ 真最优**，必须 walk-forward 验证泛化能力
5. A 股 5/20 是主流短周期；20/60 适合大波段

### 🔮 下节 L12：海龟（唐奇安通道 + ATR 仓位）
原版海龟用 ATR 动态调仓——这是 Phase 1 完全跳过的「仓位管理」入门。